## Set Price regime

In [29]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

خواندن خروجی مرحله ی قبل

In [30]:
df = pd.read_feather("../Outputs/01_df.feather")

In [34]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
df["cat2_slug"].unique()

In [ ]:
df[df["price_regime"]=="sale_unknown"]

In [ ]:

# ۱. تعریف ماسک‌های اصلی برای فیلتر کردن مسکونی و تجاری (فروش و اجاره)
is_sell = df["cat2_slug"].isin(["commercial-sell", "residential-sell"])
is_rent = df["cat2_slug"].isin(["commercial-rent", "residential-rent"])

# فیلتر کردن کل دیتافریم به طوری که فقط شامل این دسته‌ها باشد
df_filtered = df[is_sell | is_rent].copy()

# به روز رسانی ماسک‌ها بر اساس دیتای فیلتر شده جدید
is_sell = df_filtered["cat2_slug"].isin(["commercial-sell", "residential-sell"])
is_rent = df_filtered["cat2_slug"].isin(["commercial-rent", "residential-rent"])

# ۲. تعریف متغیرهای کمکی برای مقادیر قیمتی
has_price = df_filtered["price_value"].notna() & (df_filtered["price_value"] > 0)
has_rent = df_filtered["rent_value"].notna() & (df_filtered["rent_value"] > 0)
has_credit = df_filtered["credit_value"].notna() & (df_filtered["credit_value"] > 0)

# ۳. شرط‌های نامعتبر بودن (Invalid Flags)
# الف) آگهی فروش است ولی مقادیر اجاره یا رهن پر شده‌اند
invalid_sell_has_rent = is_sell & (
    (df_filtered["rent_value"].fillna(0) > 0) | 
    (df_filtered["credit_value"].fillna(0) > 0) |
    df_filtered["rent_mode"].notna() | 
    df_filtered["credit_mode"].notna()
)

# ب) آگهی اجاره است ولی قیمت فروش پر شده است
invalid_rent_has_price = is_rent & (
    (df_filtered["price_value"].fillna(0) > 0) | 
    df_filtered["price_mode"].notna()
)

# ترکیب شرایط غیرمجاز
invalid_condition = invalid_sell_has_rent | invalid_rent_has_price

# ۴. شرط‌ها و خروجی‌های بخش فروش (Sell Regime)
sell_conditions = [
    is_sell & (df_filtered["price_mode"] == "مقطوع") & has_price, 
    is_sell & (df_filtered["price_mode"] == "توافقی"), 
    is_sell & (df_filtered["price_mode"] == "مجانی"), 
    is_sell & df_filtered["price_mode"].isna() & (df_filtered["cat3_slug"] == "plot-old"),
    is_sell & df_filtered["price_mode"].isna() & (df_filtered["cat3_slug"] != "plot-old"),
    is_sell & df_filtered["price_mode"].notna() & ~df_filtered["price_mode"].isin(["مقطوع", "توافقی", "مجانی"]),
    is_sell & (df_filtered["price_mode"] == "مقطوع") & ~has_price
]

sell_choices = [
    "sale", 
    "sale_negotiable", 
    "sale_free", 
    "sale_land_no_price_mode",
    "sale_unknown", 
    "check", 
    "check"
]

# ۵. شرط‌ها و خروجی‌های بخش اجاره (Rent Regime)
rent_conditions = [
    is_rent & ((df_filtered["rent_mode"] == "توافقی") | (df_filtered["credit_mode"] == "توافقی")),
    is_rent & (df_filtered["rent_mode"] == "مجانی") & (df_filtered["credit_mode"] == "مقطوع") & has_credit,
    is_rent & (df_filtered["rent_mode"] == "مقطوع") & (df_filtered["credit_mode"] == "مقطوع") & has_rent & has_credit,
    is_rent & (df_filtered["rent_mode"] == "مقطوع") & (df_filtered["credit_mode"] == "مجانی") & has_rent,           
    is_rent & df_filtered["rent_mode"].isna() & df_filtered["credit_mode"].isna(), 
    is_rent & (df_filtered["rent_mode"] == "مجانی") & (df_filtered["credit_mode"] == "مقطوع") & ~has_credit,
    is_rent & (df_filtered["rent_mode"] == "مقطوع") & (df_filtered["credit_mode"] == "مقطوع") & ~(has_rent & has_credit),
    is_rent & (df_filtered["rent_mode"] == "مقطوع") & (df_filtered["credit_mode"] == "مجانی") & ~has_rent,
    is_rent & (df_filtered["rent_mode"].notna() | df_filtered["credit_mode"].notna()) &
        ~((df_filtered["rent_mode"] == "توافقی") | (df_filtered["credit_mode"] == "توافقی") |
        ((df_filtered["rent_mode"] == "مجانی") & (df_filtered["credit_mode"] == "مقطوع")) |
        ((df_filtered["rent_mode"] == "مقطوع") & (df_filtered["credit_mode"] == "مقطوع")) |
        ((df_filtered["rent_mode"] == "مقطوع") & (df_filtered["credit_mode"] == "مجانی")) |
        (df_filtered["rent_mode"].isna() & df_filtered["credit_mode"].isna()))
]

rent_choices = [
    "rent_negotiable", "full_morgage", "morgage_plus_rent", "rent_only", "rent_unclear",
    "check", "check", "check", "check"
]

# ۶. تجمیع شروط با اولویت اول برای مقادیر نامعتبر (Invalid)
# ابتدا شرط نامعتبر بودن چک می‌شود، اگر نبود، بقیه شرط‌ها بررسی می‌شوند
conditions = [invalid_condition] + sell_conditions + rent_conditions
choices = ["Invalid"] + sell_choices + rent_choices

# اعمال منطق روی دیتافریم فیلتر شده
df_filtered["price_regime"] = np.select(condlist=conditions, choicelist=choices, default="unknown")

# ۷. ایجاد ستون Boolean برای اعتبارسنجی نهایی
df_filtered["valid_price_regime"] = df_filtered["price_regime"] != "Invalid"

# ۸. اضافه کردن ستون دسته‌بندی کلی (Commercial / Residential)
df_filtered["property_type"] = np.where(
    df_filtered["cat2_slug"].str.contains("commercial"), 
    "Commercial", 
    "Residential"
)

# بررسی نتایج نهایی
print("توزیع سلامت داده‌ها به تفکیک کاربری ملک:")
print(df_filtered.groupby(["property_type", "valid_price_regime"]).size())

print("\nتوزیع رژیم‌های قیمتی به تفکیک کاربری ملک:")
print(
    df_filtered
    .groupby(["property_type", "price_regime"], dropna=False)
    .size()
    # .sort_values(ascending=False)
)



توزیع سلامت داده‌ها به تفکیک کاربری ملک:
property_type  valid_price_regime
Commercial     False                     38
               True                  115390
Residential    False                      9
               True                  835257
dtype: int64

توزیع رژیم‌های قیمتی به تفکیک کاربری ملک:
property_type  price_regime           
Commercial     Invalid                        38
               full_morgage                 3804
               morgage_plus_rent           70178
               rent_negotiable               534
               rent_only                    1935
               rent_unclear                   82
               sale                        31929
               sale_free                     198
               sale_negotiable              1104
               sale_unknown                 5626
Residential    Invalid                         9
               full_morgage                55418
               morgage_plus_rent          218922
               re

In [37]:
# df_commercial[df_commercial["price_regime"].isna()]

invalid_sell= df[df["price_regime"]=="rent_unclear"]
# df[
#     (df["ad_type"] == "credit") & 
#     (df["credit_mode"] != "مقطوع")
# ]
invalid_sell[
    [
        "cat2_slug",
        "title",
        # "price_mode",
        # "price_value",
        "rent_mode",
        "credit_mode",
        "rent_value",
        "credit_value",
        "description"
    ]
].head(20)

,cat2_slug,title,rent_mode,credit_mode,rent_value,credit_value,description
62291,residential-rent,نیازمندهمخانه. خانم,NaN,NaN,NaN,NaN,نیازمند همخانه خانم. بدون حاشیه ومرتب. \nلطفا اقایان تماس نگیرند. خانم های واجدشرایط فقط تماس بگیرند جهت هماهنگی. \nمحدوده خانه. خیابان فاضل
69562,residential-rent,همخونه,NaN,NaN,NaN,NaN,همخونه میخوام
84170,residential-rent,خانه اجاره ای در دل آذر داده میشود,NaN,NaN,NaN,NaN,زیر زمین به اتباع داده می‌شود\nوعدیه ۱۰۰ میلیون اجاره ۴میلیون
102985,residential-rent,سه خوابه /دو نبش(سازه خاص),NaN,NaN,NaN,NaN,قابل توجه کسانی که تلفیقی از نور و نقشه رو یکجا میخواهند❌\n\n\nآشپزخانه جزیره✔️\nآشپزخانه کثیف✔️\nصفحه کورین✔️\nکابینت نئو کلاسیک✔️\n\n\nنورگیر سرتاسری☑️\nسقف بلند☑️\nدو بالکن کاربردی☑️\nنقشه فوق تصور☑️\n\n\nیک‌خواب مستر✅\nبدون دیوار مشترک✅\n۲ پارکینگ سندی✅\n\nفرعی دنج❇️\nدسترسی عالی❇️\n\n\nاولویت با اولین تماس☎️\n\n\nاملاک کوروش\nمشاور امورملکی شما: ماهان〽️
176778,residential-rent,۱۲۰متر دوخوابه صفر,NaN,NaN,NaN,NaN,باسلام \nمنزل دوخوابه صفر کف سرامیک...شیک وتمیز ،دسترسی آسان به سوپرمارکت ونانوایی و....واقع درخیابان اشراق جنوبی خیابان ۱۵خرداد رهن کامل داده میشود
236013,residential-rent,همخونه‌میخوام,NaN,NaN,NaN,NaN,همخونه‌‌خوب‌با‌اخلاق‌باشد
277771,residential-rent,خونه دارم همخونه میخوام,NaN,NaN,NaN,NaN,مشخصات ارسال کنین تماس میگیرم. خونه با تمامی لازم موجوده. همخونه برای مادرم
392574,residential-rent,همخانه‌,NaN,NaN,NaN,NaN,اقا‌هستم‌۴۹‌سال
445240,residential-rent,راهنمایی و نیازمند همخونه,NaN,NaN,NaN,NaN,هم خونه ، همخانه ، آقا مجرد ، مرد هستم
477567,residential-rent,همخانه,NaN,NaN,NaN,NaN,به یک نفر نیاز دارم موقتی


In [38]:
df.to_feather("../Outputs/02_df.feather")
